In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import h5py
import numpy as np
import keras

# ── 1. Build the backbone (no custom weights) ──────────────────────────────
def get_bare_backbone():
    base = keras.applications.ResNet50(
        include_top=False, weights=None, input_shape=(None, None, 3)
    )
    for layer in base.layers:
        if 'conv4_block1_0_conv' in layer.name or 'conv4_block1_1_conv' in layer.name:
            layer.strides = (1, 1)
        if 'conv4_block' in layer.name and '_2_conv' in layer.name:
            layer.dilation_rate = (2, 2)
            layer.padding = 'same'
    output = base.get_layer("conv4_block6_out").output
    backbone = keras.Model(inputs=base.input, outputs=output, name="resnet50_conv4_backbone")
    return backbone

src_path = '/kaggle/input/models/ramakrishnasastry05/dialated-model/tensorflow2/default/1/all_datasets_backbone_dialated.h5'

2026-05-01 18:46:56.796691: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777661217.008663      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777661217.075634      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777661217.578581      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777661217.578631      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777661217.578635      23 computation_placer.cc:177] computation placer alr

In [3]:
import h5py
import keras

# Build backbone
backbone = get_bare_backbone()

with h5py.File(src_path, 'r') as f:
    saved_layers = sorted(
        f['layers'].keys(),
        key=lambda x: (x.rsplit('_', 1)[0], int(x.rsplit('_', 1)[-1]) if x.rsplit('_', 1)[-1].isdigit() else 0)
    )
    saved_weight_layers = []
    for layer_name in saved_layers:
        vars_group = f[f'layers/{layer_name}/vars']
        if len(vars_group) > 0:
            shapes = [vars_group[str(i)].shape for i in range(len(vars_group))]
            saved_weight_layers.append((layer_name, shapes))

model_weight_layers = [(l.name, [w.shape for w in l.get_weights()]) for l in backbone.layers if l.get_weights()]

print(f"{'IDX':<5} {'SAVED NAME':<35} {'SAVED SHAPES':<45} {'MODEL NAME':<35} {'MODEL SHAPES'}")
print("-" * 160)
for i, ((sname, sshapes), (mname, mshapes)) in enumerate(zip(saved_weight_layers, model_weight_layers)):
    match = "✅" if sshapes == mshapes else "❌"
    print(f"{i:<5} {sname:<35} {str(sshapes):<45} {mname:<35} {str(mshapes)}  {match}")

I0000 00:00:1777661242.118810      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777661242.124902      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


IDX   SAVED NAME                          SAVED SHAPES                                  MODEL NAME                          MODEL SHAPES
----------------------------------------------------------------------------------------------------------------------------------------------------------------
0     batch_normalization                 [(64,), (64,), (64,), (64,)]                  conv1_conv                          [(7, 7, 3, 64), (64,)]  ❌
1     batch_normalization_1               [(64,), (64,), (64,), (64,)]                  conv1_bn                            [(64,), (64,), (64,), (64,)]  ✅
2     batch_normalization_2               [(64,), (64,), (64,), (64,)]                  conv2_block1_1_conv                 [(1, 1, 64, 64), (64,)]  ❌
3     batch_normalization_3               [(256,), (256,), (256,), (256,)]              conv2_block1_1_bn                   [(64,), (64,), (64,), (64,)]  ❌
4     batch_normalization_4               [(256,), (256,), (256,), (256,)]              c

In [4]:
import h5py
import numpy as np
import keras

src_path = '/kaggle/input/models/ramakrishnasastry05/dialated-model/tensorflow2/default/1/all_datasets_backbone_dialated.h5'

backbone = get_bare_backbone()

# ── 1. Read all saved weights grouped by type ──────────────────────────────
with h5py.File(src_path, 'r') as f:
    saved_layers = sorted(
        f['layers'].keys(),
        key=lambda x: (x.rsplit('_', 1)[0], int(x.rsplit('_', 1)[-1]) if x.rsplit('_', 1)[-1].isdigit() else 0)
    )
    
    saved_bn_weights   = []  # 4 vars each: gamma, beta, mean, var
    saved_conv_weights = []  # 2 vars each: kernel, bias

    for layer_name in saved_layers:
        vars_group = f[f'layers/{layer_name}/vars']
        if len(vars_group) == 0:
            continue
        weights = [vars_group[str(i)][()] for i in range(len(vars_group))]
        if layer_name.startswith('batch_normalization'):
            saved_bn_weights.append(weights)
        elif layer_name.startswith('conv2d'):
            saved_conv_weights.append(weights)

print(f"Saved BN layers  : {len(saved_bn_weights)}")
print(f"Saved Conv layers: {len(saved_conv_weights)}")

# ── 2. Walk model layers and assign from the correct pool ──────────────────
bn_idx   = 0
conv_idx = 0

for layer in backbone.layers:
    if not layer.get_weights():
        continue
    
    layer_type = type(layer).__name__

    if 'BatchNormalization' in layer_type:
        if bn_idx >= len(saved_bn_weights):
            print(f"⚠️  Ran out of BN weights at layer {layer.name}")
            break
        layer.set_weights(saved_bn_weights[bn_idx])
        bn_idx += 1

    elif 'Conv2D' in layer_type:
        if conv_idx >= len(saved_conv_weights):
            print(f"⚠️  Ran out of Conv weights at layer {layer.name}")
            break
        layer.set_weights(saved_conv_weights[conv_idx])
        conv_idx += 1

print(f"✅ Assigned {bn_idx} BN + {conv_idx} Conv layers")

# ── 3. Save in standard format ─────────────────────────────────────────────
out_path = '/kaggle/working/backbone_compatible.weights.h5'
backbone.save_weights(out_path)
print(f"✅ Saved to: {out_path}")

# ── 4. Verify round-trip ───────────────────────────────────────────────────
backbone2 = get_bare_backbone()
backbone2.load_weights(out_path)

mismatches = 0
for l1, l2 in zip(
    [l for l in backbone.layers if l.get_weights()],
    [l for l in backbone2.layers if l.get_weights()]
):
    for w1, w2 in zip(l1.get_weights(), l2.get_weights()):
        if not np.allclose(w1, w2):
            print(f"❌ Mismatch in {l1.name}")
            mismatches += 1

if mismatches == 0:
    print("✅ Verification passed — all weights match perfectly")

Saved BN layers  : 43
Saved Conv layers: 43
✅ Assigned 43 BN + 43 Conv layers
✅ Saved to: /kaggle/working/backbone_compatible.weights.h5
✅ Verification passed — all weights match perfectly


In [5]:
import h5py

path = '/kaggle/input/models/ramakrishnasastry05/dialated-model/tensorflow2/default/1/all_datasets_backbone_dialated.h5'

with h5py.File(path, 'r') as f:
    def print_structure(name, obj):
        print(name)
    f.visititems(print_structure)
    print("\nTop-level keys:", list(f.keys()))

layers
layers/activation
layers/activation/vars
layers/activation_1
layers/activation_1/vars
layers/activation_10
layers/activation_10/vars
layers/activation_11
layers/activation_11/vars
layers/activation_12
layers/activation_12/vars
layers/activation_13
layers/activation_13/vars
layers/activation_14
layers/activation_14/vars
layers/activation_15
layers/activation_15/vars
layers/activation_16
layers/activation_16/vars
layers/activation_17
layers/activation_17/vars
layers/activation_18
layers/activation_18/vars
layers/activation_19
layers/activation_19/vars
layers/activation_2
layers/activation_2/vars
layers/activation_20
layers/activation_20/vars
layers/activation_21
layers/activation_21/vars
layers/activation_22
layers/activation_22/vars
layers/activation_23
layers/activation_23/vars
layers/activation_24
layers/activation_24/vars
layers/activation_25
layers/activation_25/vars
layers/activation_26
layers/activation_26/vars
layers/activation_27
layers/activation_27/vars
layers/activatio

In [6]:
# !git clone https://github.com/RamaKrishnaSastry/MajorProject.git
# %cd MajorProject
# !pip install -r requirements.txt

!git clone -b feat-gradcam --single-branch https://github.com/RamaKrishnaSastry/MajorProject.git
%cd MajorProject
!pip install -r requirements.txt

Cloning into 'MajorProject'...
remote: Enumerating objects: 588, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 588 (delta 194), reused 205 (delta 184), pack-reused 350 (from 1)
Receiving objects: 100% (588/588), 3.02 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (355/355), done.
/kaggle/working/MajorProject


In [7]:
# !python test_ordinal_weighted_loss.py
# !python test_qwk_calculation.py
# !python test_qwk_multioutput.py

In [8]:
# !python main_pipeline.py --mock --epochs 10 --single-stage --output-dir /kaggle/working

In [9]:
import os
import shutil
import pandas as pd

# =========================
# CONFIG (EDIT THESE PATHS)
# =========================

# Root Kaggle extracted folder
RAW_ROOT = "/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset"

# Input paths
IMAGE_SRC = os.path.join(
    RAW_ROOT,
    "B.%20Disease%20Grading",
    "B. Disease Grading",
    "1. Original Images",
    "a. Training Set"
)

CSV_SRC = os.path.join(
    RAW_ROOT,
    "B.%20Disease%20Grading",
    "B. Disease Grading",
    "2. Groundtruths",
    "a. IDRiD_Disease Grading_Training Labels.csv"
)

# Output paths
OUTPUT_DIR = "/kaggle/working/"
IMAGE_DST = os.path.join(OUTPUT_DIR, "images")
CSV_DST = os.path.join(OUTPUT_DIR, "DME_Grades.csv")

# Choose label type: "DR" or "DME"
LABEL_TYPE = "DR"   # change to "DME" if needed

# =========================
# STEP 1: CREATE FOLDERS
# =========================

os.makedirs(IMAGE_DST, exist_ok=True)

# =========================
# STEP 2: COPY IMAGES
# =========================

print("Copying images...")

for file in os.listdir(IMAGE_SRC):
    if file.lower().endswith(".jpg"):
        src_path = os.path.join(IMAGE_SRC, file)
        dst_path = os.path.join(IMAGE_DST, file)
        shutil.copy2(src_path, dst_path)

print(f"Images copied to: {IMAGE_DST}")

# =========================
# STEP 3: PROCESS CSV
# =========================

print("Processing CSV...")

df = pd.read_csv(CSV_SRC)

# Clean column names (VERY IMPORTANT)
df.columns = [col.strip() for col in df.columns]

# Detect columns safely
image_col = None
dr_col = None
dme_col = None

for col in df.columns:
    if "Image" in col:
        image_col = col
    elif "Retinopathy" in col:
        dr_col = col
    elif "macular" in col.lower():
        dme_col = col

# Select label
# if LABEL_TYPE == "DR":
#     label_col = dr_col
# elif LABEL_TYPE == "DME":
#     label_col = dme_col
# else:
#     raise ValueError("LABEL_TYPE must be 'DR' or 'DME'")

# =========================
# STEP 4: CREATE FINAL CSV
# =========================

new_df = pd.DataFrame({
    "Image name": df[image_col].astype(str) + ".jpg",
    "Retinopathy grade": df[dr_col],
    "Risk of macular edema": df[dme_col]
})

# Save
new_df.to_csv(CSV_DST, index=False)

print(f"Final CSV saved at: {CSV_DST}")

# =========================
# DONE
# =========================

print("\n✅ Dataset ready!")
print(f"Images folder: {IMAGE_DST}")
print(f"CSV file: {CSV_DST}")

Copying images...
Images copied to: /kaggle/working/images
Processing CSV...
Final CSV saved at: /kaggle/working/DME_Grades.csv

✅ Dataset ready!
Images folder: /kaggle/working/images
CSV file: /kaggle/working/DME_Grades.csv


In [10]:
# import os
# import shutil
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # =========================
# # CONFIGURATION
# # =========================

# RAW_ROOT = "/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset"
# OUTPUT_DIR = "/kaggle/working/"

# # Image paths
# TRAIN_IMG_SRC = os.path.join(
#     RAW_ROOT,
#     "B.%20Disease%20Grading",
#     "B. Disease Grading",
#     "1. Original Images",
#     "a. Training Set"
# )

# TEST_IMG_SRC = os.path.join(
#     RAW_ROOT,
#     "B.%20Disease%20Grading",
#     "B. Disease Grading",
#     "1. Original Images",
#     "b. Testing Set"
# )

# # CSV paths
# TRAIN_CSV_SRC = os.path.join(
#     RAW_ROOT,
#     "B.%20Disease%20Grading",
#     "B. Disease Grading",
#     "2. Groundtruths",
#     "a. IDRiD_Disease Grading_Training Labels.csv"
# )

# TEST_CSV_SRC = os.path.join(
#     RAW_ROOT,
#     "B.%20Disease%20Grading",
#     "B. Disease Grading",
#     "2. Groundtruths",
#     "b. IDRiD_Disease Grading_Testing Labels.csv"
# )

# # Output folders
# TRAIN_DST = os.path.join(OUTPUT_DIR, "train_images")
# TEST_DST = os.path.join(OUTPUT_DIR, "test_images")

# os.makedirs(TRAIN_DST, exist_ok=True)
# os.makedirs(TEST_DST, exist_ok=True)

# # =========================
# # STEP 1: LOAD AND CLEAN CSVs
# # =========================

# def load_and_clean(csv_path):
#     df = pd.read_csv(csv_path)
#     df.columns = [col.strip() for col in df.columns]

#     image_col, dr_col, dme_col = None, None, None

#     for col in df.columns:
#         if "Image" in col:
#             image_col = col
#         elif "Retinopathy" in col:
#             dr_col = col
#         elif "macular" in col.lower():
#             dme_col = col

#     cleaned_df = pd.DataFrame({
#         "Image name": df[image_col].astype(str) + ".jpg",
#         "Retinopathy grade": df[dr_col],
#         "Risk of macular edema": df[dme_col]
#     })

#     return cleaned_df

# train_df = load_and_clean(TRAIN_CSV_SRC)
# test_df = load_and_clean(TEST_CSV_SRC)

# # =========================
# # STEP 2: MOVE 20% OF TEST TO TRAIN POOL
# # =========================

# test_to_train, remaining_test = train_test_split(
#     test_df,
#     test_size=0.2,
#     stratify=test_df["Retinopathy grade"],  # maintain DR distribution
#     random_state=42
# )

# # Combine with original training data
# combined_df = pd.concat([train_df, test_to_train], ignore_index=True)

# # =========================
# # STEP 3: FINAL 80-20 TRAIN-TEST SPLIT
# # =========================

# train_final, test_final = train_test_split(
#     combined_df,
#     test_size=0.1,
#     stratify=combined_df["Retinopathy grade"],
#     random_state=42
# )

# print(f"Final Training Samples: {len(train_final)}")
# print(f"Final Testing Samples: {len(test_final)}")

# # =========================
# # STEP 4: COPY IMAGES
# # =========================

# def copy_images(df, src_train, src_test, dst_folder):
#     for img_name in df["Image name"]:
#         src_path = os.path.join(src_train, img_name)
#         if not os.path.exists(src_path):
#             src_path = os.path.join(src_test, img_name)

#         if os.path.exists(src_path):
#             shutil.copy2(src_path, os.path.join(dst_folder, img_name))
#         else:
#             print(f"Warning: {img_name} not found.")

# print("Copying training images...")
# copy_images(train_final, TRAIN_IMG_SRC, TEST_IMG_SRC, TRAIN_DST)

# print("Copying testing images...")
# copy_images(test_final, TRAIN_IMG_SRC, TEST_IMG_SRC, TEST_DST)

# # =========================
# # STEP 5: SAVE FINAL CSVs
# # =========================

# train_csv_path = os.path.join(OUTPUT_DIR, "train_labels.csv")
# test_csv_path = os.path.join(OUTPUT_DIR, "test_labels.csv")

# train_final.to_csv(train_csv_path, index=False)
# test_final.to_csv(test_csv_path, index=False)

# # =========================
# # STEP 6: DATASET FOR MODEL
# # =========================

# # Dataset specifically for training the model
# model_dataset = train_final.copy()
# model_dataset["image_path"] = model_dataset["Image name"].apply(
#     lambda x: os.path.join(TRAIN_DST, x)
# )

# model_dataset_path = os.path.join(OUTPUT_DIR, "model_training_dataset.csv")
# model_dataset.to_csv(model_dataset_path, index=False)

# # =========================
# # SUMMARY
# # =========================

# print("\n✅ Dataset preparation complete!")
# print(f"Training Images Folder: {TRAIN_DST}")
# print(f"Testing Images Folder: {TEST_DST}")
# print(f"Training CSV: {train_csv_path}")
# print(f"Testing CSV: {test_csv_path}")
# print(f"Model Training Dataset CSV: {model_dataset_path}")

In [11]:
!python main_pipeline.py \
  --csv /kaggle/working/DME_Grades.csv \
  --image-dir /kaggle/working/images \
  --config config.yaml \
  --long-stage2\
  --output-dir /kaggle/working/output \
  --use-eyepacs /kaggle/working/backbone_compatible.weights.h5 \
  --save-images-path /kaggle/working/split_images
  # --use-eyepacs /kaggle/input/models/ramakrishnasastryp/all-datasets-backbone/tensorflow2/default/1/all_datasets_backbone.weights.h5
  # --single-stage

INFO: Loaded config from 'config.yaml'.
2026-05-01 18:47:41.294115: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777661261.316448     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777661261.324485     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777661261.343834     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777661261.343864     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777661261.343867     106 computati

In [12]:
!python test_cross_dataset_dr.py \
  --use-model \
  --use-weights '/kaggle/working/output/model_stage2.model.h5' \
  --messidor-images '/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Images/b. Testing Set' \
  --messidor-csv '/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv'

2026-05-01 21:56:35.007970: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777672595.034984    2787 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777672595.043000    2787 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777672595.065534    2787 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672595.065590    2787 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672595.065605    2787 computation_placer.cc:177] computation placer alr

In [13]:
!python test_cross_dataset_dr.py \
  --use-model \
  --use-weights '/kaggle/working/output/model_stage2.model.h5' \
  --messidor-images '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess' \
  --messidor-csv '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'

2026-05-01 21:57:58.781774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777672678.806563    2885 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777672678.814777    2885 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777672678.835169    2885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672678.835212    2885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672678.835216    2885 computation_placer.cc:177] computation placer alr

In [14]:
!python test_cross_dataset_dr.py \
  --use-model \
  --use-weights '/kaggle/working/output/model_stage2.model.h5' \
  --messidor-images '/kaggle/input/datasets/mariaherrerot/aptos2019/test_images/test_images' \
  --messidor-csv '/kaggle/input/datasets/mariaherrerot/aptos2019/test.csv'

2026-05-01 22:00:33.119451: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777672833.141910    2976 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777672833.149402    2976 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777672833.168023    2976 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672833.168052    2976 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672833.168056    2976 computation_placer.cc:177] computation placer alr

In [15]:
# !python test_cross_dataset_dr.py \
#   --use-model \
#   --use-weights '/kaggle/working/output/model_stage2.model.h5' \
#   --messidor-images '/kaggle/input/datasets/mariaherrerot/aptos2019/train_images/train_images' \
#   --messidor-csv '/kaggle/input/datasets/mariaherrerot/aptos2019/train_1.csv'

In [16]:
!python test_cross_dataset_dr.py \
  --use-model \
  --use-weights '/kaggle/working/output/model_stage2.model.h5' \
  --messidor-images '/kaggle/input/datasets/mariaherrerot/aptos2019/val_images/val_images' \
  --messidor-csv '/kaggle/input/datasets/mariaherrerot/aptos2019/valid.csv'

2026-05-01 22:02:36.620481: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777672956.648025    3074 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777672956.657091    3074 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777672956.679242    3074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672956.679306    3074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672956.679310    3074 computation_placer.cc:177] computation placer alr

In [17]:
!python /kaggle/working/MajorProject/bootstrap_confidence_intervels.py \
        --model  '/kaggle/working/output/model_stage2.model.h5' \
        --csv    '/kaggle/working/split_images/val_csv/val_split.csv' \
        --imgdir '/kaggle/working/split_images/val_images'\
        --out    '/kaggle/wprking/bootstrap_results'

2026-05-01 22:04:46.241003: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777673086.284358    3172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777673086.293774    3172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777673086.316692    3172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777673086.316742    3172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777673086.316747    3172 computation_placer.cc:177] computation placer alr

In [18]:
!python /kaggle/working/MajorProject/gradcam_visualization.py \
    --model '/kaggle/working/output/model_stage2.model.h5' \
    --csv   '/kaggle/working/split_images/val_csv/val_split.csv' \
    --imgdir '/kaggle/working/split_images/val_csv/val_images' \
    --out   '/kaggle/working/gradcam_outputs' \
    --n     8

E0000 00:00:1777673123.826058    3296 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777673123.834234    3296 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777673123.853679    3296 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777673123.853708    3296 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777673123.853712    3296 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777673123.853716    3296 computation_placer.cc:177] computation placer already registered. Please check linka

In [19]:
!python /kaggle/working/MajorProject/expanded_comparison_table.py --out 'kaggle/working/comparison_outputs'


  Generating comparison table (12 methods) …

  Text table: kaggle/working/comparison_outputs/comparison_table.txt
  LaTeX table: kaggle/working/comparison_outputs/comparison_table_latex.tex
  CSV table: kaggle/working/comparison_outputs/comparison_table.csv
  Chart: kaggle/working/comparison_outputs/comparison_chart.png

  ✅ Done.



In [20]:
# import os
# import glob
# import json
# from main_pipeline import (
#     load_config,
#     set_global_seed,
#     stage_data_preparation,
#     stage_training,
#     stage_evaluation,
# )

# csv_path = "/kaggle/working/DME_Grades.csv"
# image_dir = "/kaggle/working/images"

# cfg = load_config("config.yaml")
# cfg["output_dir"] = "/kaggle/working"
# cfg["checkpoint_dir"] = os.path.join(cfg["output_dir"], "checkpoints")

# set_global_seed(cfg.get("seed", 42))
# train_ds, val_ds, class_weights, _ = stage_data_preparation(csv_path, image_dir, cfg)

# stage1_dir = os.path.join(cfg["checkpoint_dir"], "stage1")
# snapshots = sorted(glob.glob(os.path.join(stage1_dir, "stage1_best_*.weights.h5")))
# init_weights = snapshots[-1] if snapshots else os.path.join(stage1_dir, "best_qwk.weights.h5")
# if not os.path.exists(init_weights):
#     raise FileNotFoundError(f"Stage1 checkpoint not found: {init_weights}")

# metrics_path = os.path.join(cfg["output_dir"], "eval_stage1", "comprehensive_metrics.json")
# with open(metrics_path, "r") as f:
#     stage1_qwk = float(json.load(f)["qwk"])

# model, history2, weights2 = stage_training(
#     train_ds=train_ds,
#     val_ds=val_ds,
#     class_weights=class_weights,
#     config=cfg,
#     stage_name="stage2",
#     pretrained_weights=init_weights,
#     stage1_baseline_qwk=stage1_qwk,
# )
# metrics2 = stage_evaluation(model, val_ds, cfg, stage_name="stage2")
# print(f"Stage2 done. QWK={metrics2.get('qwk', float('nan')):.4f}")
# PY

In [21]:
# !python main_pipeline.py \
#   --csv /kaggle/working/DME_Grades.csv \
#   --image-dir /kaggle/working/images \
#   --config config.yaml \
#   --output-dir /kaggle/working/

In [22]:
# 1. Download IDRiD (already have it?)
# Path: data/IRDID/images + data/IRDID/DME_Grades.csv

# 2. Run single epoch smoke test
# !python main_pipeline.py \
#   --csv /kaggle/working/DME_Grades.csv \
#   --image-dir /kaggle/working/images \
#   --config config.yaml \
#   --epochs 3 \
#   --single-stage \
#   --output-dir /kaggle/working/


# !python main_pipeline.py \
#   --csv /path/to/DME_Grades.csv \
#   --image-dir /path/to/images \
#   --config config.yaml \
#   --single-stage

# python main_pipeline.py \
#   --csv data/IRDID/DME_Grades.csv \
#   --image-dir data/IRDID/images \
#   --config config.yaml \
#   --output-dir results/ \
#   --epochs 30

# 3. Check epoch 1 output for:
#    - QWK should be > 0.0 (not stuck at 0)
#    - Predictions distributed across classes (not all class 3)
#    - MAE < 2.0 (reasonable ordinal error)

In [23]:
import yaml; print(yaml.dump(yaml.safe_load(open('config.yaml')), default_flow_style=False))

ablation:
  collapse_guard_enabled: true
  collapse_guard_patience: 2
  collapse_guard_ratio: 0.95
  collapse_guard_restarts: 1
  collapse_guard_start_epoch: 2
  epochs_per_model: 5
  n_bootstrap: 500
  output_dir: ablation_results
augment_train: true
batch_size: 8
border_fraction: 0.1
checkpoint_dir: pipeline_outputs/checkpoints
clip_limit: 2.0
dme_class_weight_clip_ratio: 7.0
evaluation:
  calibrate_dme_thresholds: true
  calibrate_dr_thresholds: true
  calibration_min_qwk_gain: 0.001
  dr_calibration_max_accuracy_drop: 0.0
  output_visualisations: true
  qwk_target: 0.8
  save_boundary_confusion: true
  save_confusion_matrix: true
  save_per_class_metrics: true
grid_size: 8
input_shape:
- 512
- 512
- 3
joint_selection:
  dme_floor: 0.7
  enabled: true
  fallback_step: 0.02
  min_threshold: 0.6
  thresholds:
  - - 0.7
    - 0.8
  - - 0.72
    - 0.78
  - - 0.75
    - 0.75
  - - 0.7
    - 0.75
  - - 0.7
    - 0.72
  - - 0.7
    - 0.7
  - - 0.68
    - 0.7
long_stage2_mode: true
max_batc

In [24]:
import subprocess

# Check EXACTLY which file is running and what's in the critical sections
checks = [
    ['grep', '-n', 'weight = distance\|weight = 1.0 + distance', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'entropy_weight', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'dr_output.*0\\.', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'focal_loss_gamma', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'clipnorm', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'smoothing', '/kaggle/working/MajorProject/train_enhanced.py'],
    ['grep', '-n', 'learning_rate', '/kaggle/working/MajorProject/config.yaml'],
    ['grep', '-n', 'focal_loss_gamma', '/kaggle/working/MajorProject/config.yaml'],
    ['grep', '-n', 'oversample\|oversampling', '/kaggle/working/MajorProject/dataset_loader_advanced.py'],
]

labels = ['ordinal matrix', 'entropy term', 'dr loss weight', 'focal gamma in py',
          'clipnorm', 'label smoothing', 'lr in config', 'focal gamma in config', 'oversampling']

for label, cmd in zip(labels, checks):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"\n=== {label} ===")
    print(r.stdout if r.stdout else "(nothing found)")


=== ordinal matrix ===
190:                    weight = 1.0 + distance


=== entropy term ===
(nothing found)

=== dr loss weight ===
(nothing found)

=== focal gamma in py ===
91:    "focal_loss_gamma": 2.0,
162:        focal_loss_gamma=2.0,
169:        self.focal_loss_gamma = focal_loss_gamma
203:        if self.focal_loss_gamma > 0:
205:            focal_factor = tf.pow(1.0 - p_t, self.focal_loss_gamma)
232:            "focal_loss_gamma":       float(getattr(self, "focal_loss_gamma", 0.0)),
840:    focal_loss_gamma: float = 2.0,
852:            focal_loss_gamma=focal_loss_gamma,
857:        logger.info("Focal loss gamma=%.1f.", focal_loss_gamma)
1374:        focal_loss_gamma=cfg.get("focal_loss_gamma", 2.0),


=== clipnorm ===
847:    optimizer = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0)


=== label smoothing ===
92:    "dme_label_smoothing": 0.05,
163:        label_smoothing=0.05,
170:        self.label_smoothing = float(label_smoothing)
197:        if self.

<>:5: SyntaxWarning: invalid escape sequence '\|'
<>:13: SyntaxWarning: invalid escape sequence '\|'
<>:5: SyntaxWarning: invalid escape sequence '\|'
<>:13: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_23/2344279109.py:5: SyntaxWarning: invalid escape sequence '\|'
  ['grep', '-n', 'weight = distance\|weight = 1.0 + distance', '/kaggle/working/MajorProject/train_enhanced.py'],
/tmp/ipykernel_23/2344279109.py:13: SyntaxWarning: invalid escape sequence '\|'
  ['grep', '-n', 'oversample\|oversampling', '/kaggle/working/MajorProject/dataset_loader_advanced.py'],


In [25]:
import subprocess
r = subprocess.run(['grep', '-n', 
    'stage2\|pretrained_weights\|best_qwk\|weights1\|stage1.*weights\|checkpoint',
    '/kaggle/working/MajorProject/main_pipeline.py'],
    capture_output=True, text=True)
print(r.stdout)

6:- Automatic model selection and checkpointing
68:    "long_stage2_mode": True,
97:    "stage2": {
112:        "stage2_freeze_aspp_bn": True,
113:        "stage2_checkpoint_use_stage1_baseline": True,
120:        "stage2_revert_if_worse": True,
121:        "stage2_min_improvement": 0.003,
124:    "checkpoint_dir": "pipeline_outputs/checkpoints",
128:    # Joint DME+DR selection policy for checkpoints and final stage ranking.
397:    pretrained_weights: Optional[str] = None,
415:        Stage key in config (``"stage1"`` or ``"stage2"``).
416:    pretrained_weights : str, optional
419:        Stage 1 best QWK used by stage2 collapse guard.
430:        ``(model, history, output_weights, selected_checkpoint_path)``
435:    long_stage2_mode = bool(config.get("long_stage2_mode", True))
437:    checkpoint_dir = os.path.join(config.get("checkpoint_dir", "checkpoints"), stage_name)
438:    os.makedirs(checkpoint_dir, exist_ok=True)
440:    if stage_name == "stage2":
441:        if long_stage2_

<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:3: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_23/4181838838.py:3: SyntaxWarning: invalid escape sequence '\|'
  'stage2\|pretrained_weights\|best_qwk\|weights1\|stage1.*weights\|checkpoint',
